<a href="https://colab.research.google.com/github/mikahamelin-gif/IS-4487-final-project-nvidia/blob/main/IS_4487_Final_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Read Me:
- Before working, click the arrow by "run all" and select "restart session and run all" to wipe the history and run all cells in order
- Commit changes to GitHub by saving after ***every time*** you edit it

# IS 4487 Final Project - NVIDIA

### Mikayla Hamelin, Beck Feely, Nathaniel Elbaum, Billy Ortiz


### Summary of business problem:

NVIDIA has to order chips from its manufacturing partners months before it sells them, while demand from AI data centers is growing fast and hard to predict. If it underestimates demand, it misses sales and customers turn to competitors. If it overestimates, it’s left with costly inventory it can’t sell. Our project forecasts how many AI chips NVIDIA will ship next quarter, measured in units or H100 equivalents, so it can plan production and capacity ahead of time.


## Business Problem & Context

### Description of the organization and the business challenge:

AI is the world's fastest growing market and technology, and this boom is supported largely by one company's technology: NVIDIA. NVIDIA designs graphics processing units (GPUs) and the software that runs them, and its chips have become the main hardware used to train and run AI models becuase of their ability to perform parallel processing, or runinng thousands of calculations at once. NVIDIA orders the chips they design from manufacturing partners months in advance. The challenge is predicting how many chips it will need when AI demand is growing quickly and changing from quarter to quarter.

### Why this problem matters (business impact)

Because of the nature of this industry, NVIDIA is in a precarious position. If NVIDIA orders too few chips, it misses sales and risks pushing its key buyers (cloud providers, AI companies, and other data center operators) toward competitors like AMD or toward building their own chips, as Google and Amazon already do. NVIDIA leads the AI chip market, so many other companies' operations and valuations depend on its supply, and some would argue the AI industry as a whole does too. If it orders too many, it ends up with expensive inventory that loses value as newer chips come out. Since production space has to be reserved well ahead of time, a better forecast helps NVIDIA plan capacity, manage costs, and keep its biggest customers and the broader AI industry supplied.

###Brief description of relevant industry or market factors
*  **Big tech spending**: Most demand comes from a few large companies like Microsoft,Google, Amazon and Meta, so changes in their data center budgets move NVIDIA’s sales.
*  **Supply limits**: Manufacturing capacity is limited and can’t be expanded quickly.
*   **Competition**: AMD, plus companies like Google and Amazon designing their own AI chips, could take share.
*   **Regulation and infrastructure**: U.S. export restrictions limit sales to some countries, and data centers need large amounts of power, which can slow how fast new ones get built.

## Data Sources & References

### Links to datasets used (original sources, not just download locations), & brief explanation of contents,

* https://epoch.ai/data-insights/hyperscaler-capex-trend -- aggregate data of capital expenditure since 2022 by the main players in the AI race. Epoch, in preparing the dataset, already accounted for the differences in how each company reports CAPEX spending and normalized it between the companies.

* https://www.sec.gov/edgar/browse/?CIK=0001045810 - SEC quarterly financial filings by NVIDIA as required because they're a public company. Data aggregated by Our World in Data here: https://ourworldindata.org/grapher/nvidia-quarterly-revenue-segment


* https://www.census.gov/construction/c30/historical_data.html -- historical and current data of public and private construction spend in various industries like office, commercial, etc. When datacenters are built, GPU orders aren't far behind.


### Citation of at least two credible references for the business context


*  NVIDIA Blog Announcing their partnership with TSMC in Arizona: https://blogs.nvidia.com/blog/nvidia-manufacture-american-made-ai-supercomputers-us/
*  NVIDIA SEC Report detailing their company, risks, and goals: https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm

## Data Acquisition & Loading - Nathan

In [28]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [37]:

file = "/content/NVDA_Quarterly_Demand_Drivers.xlsx"

df = pd.read_excel(file, sheet_name="AI Chip Demand Drivers")

# Remove source and status columns
df = df.loc[:, ~df.columns.astype(str).str.contains("source|status|verification|precision", case=False)]

# Comment out the line above to view sources and status information

df = df.sort_values(["Year", "Quarter Number"])
df = df.reset_index(drop=True)

print("Final Dataset:", df.shape)
display(df)


Final Dataset: (42, 36)


,Quarter,Year,Quarter Number,Quarter End,SEC Fiscal Year,NVDA Revenue ($mm),NVDA Cost of Revenue ($mm),NVDA Gross Profit ($mm),NVDA R&D Expense ($mm),NVDA SG&A Expense ($mm),...,NVDA Free Cash Flow ($mm),NVDA Stock-Based Compensation ($mm),NVDA Data Center Revenue ($mm),NVDA Data Center / Revenue,NVDA Operating Cash Flow Margin,NVDA Capex / Revenue,NVDA Total Assets ($mm),NVDA Total Liabilities ($mm),NVDA Shareholders Equity ($mm),NVDA Balance Sheet Check ($mm)
0,2016 Q1,2016,1,2016-05-01,2017,1305,554,751,346,159,...,263,53,143,0.109579,0.243678,0.042146,7108,2912,4196,0
1,2016 Q2,2016,2,2016-07-31,2017,1428,602,826,350,157,...,169,58,151,0.105742,0.140756,0.022409,7461,2981,4480,0
2,2016 Q3,2016,3,2016-10-30,2017,2004,821,1183,373,171,...,394,65,240,0.119760,0.215569,0.018962,9612,4288,5324,0
3,2016 Q4,2016,4,2017-01-29,2017,2173,870,1303,394,176,...,670,71,296,0.136217,0.331799,0.023470,9841,4079,5762,0
4,2017 Q1,2017,1,2017-04-30,2018,1937,787,1150,411,185,...,228,76,409,0.211151,0.145586,0.027878,9410,3278,6132,0
5,2017 Q2,2017,2,2017-07-30,2018,2230,928,1302,416,198,...,651,81,416,0.186547,0.316143,0.024215,9402,3429,5973,0
6,2017 Q3,2017,3,2017-10-29,2018,2636,1067,1569,462,212,...,1088,107,501,0.190061,0.438923,0.026176,9830,3478,6352,0
7,2017 Q4,2017,4,2018-01-28,2018,2911,1110,1801,507,221,...,942,127,606,0.208176,0.466506,0.142906,11241,3770,7471,0
8,2018 Q1,2018,1,2018-04-29,2019,3207,1139,2068,542,231,...,1327,129,701,0.218584,0.450577,0.036795,11460,3743,7717,0
9,2018 Q2,2018,2,2018-07-29,2019,3123,1148,1975,581,237,...,784,132,760,0.243356,0.292347,0.041306,12882,4087,8795,0


In [38]:

print("Total Quarters:", len(df))

print("Duplicate Quarters:",
      df.duplicated(subset=["Year", "Quarter Number"]).sum())

print("Missing Values:")
print(df.isnull().sum())

print("Data Types:")
print(df.dtypes)


Total Quarters: 42
Duplicate Quarters: 0
Missing Values:
Quarter                                     0
Year                                        0
Quarter Number                              0
Quarter End                                 0
SEC Fiscal Year                             0
NVDA Revenue ($mm)                          0
NVDA Cost of Revenue ($mm)                  0
NVDA Gross Profit ($mm)                     0
NVDA R&D Expense ($mm)                      0
NVDA SG&A Expense ($mm)                     0
NVDA Operating Expenses ($mm)               0
NVDA Operating Income ($mm)                 0
NVDA Net Income ($mm)                       0
NVDA Diluted EPS Historical ($)             0
NVDA YoY Revenue Growth                     4
NVDA QoQ Revenue Growth                     1
NVDA Gross Margin                           0
NVDA Operating Margin                       0
NVDA Net Margin                             0
NVDA Opex / Revenue                         0
NVDA Cash & Equivalents

In [41]:

# Remove unnecessary columns
df = df.drop(columns=[
    "NVDA Balance Sheet Check ($mm)",
    "SEC Fiscal Year",
    "Quarter End",
    "NVDA Diluted EPS Historical ($)",
    "NVDA Opex / Revenue",
    "NVDA Capex / Revenue"
], errors="ignore")

# Remove source and documentation columns
df = df.loc[:, ~df.columns.astype(str).str.contains(
    "source|status|verification|precision", case=False
)]

# Remove empty columns
df = df.dropna(axis=1, how="all")

# Sort by quarter
df = df.sort_values(["Year", "Quarter Number"])
df = df.reset_index(drop=True)

print("Final Dataset:", df.shape)
display(df)


Final Dataset: (42, 30)


,Quarter,Year,Quarter Number,NVDA Revenue ($mm),NVDA Cost of Revenue ($mm),NVDA Gross Profit ($mm),NVDA R&D Expense ($mm),NVDA SG&A Expense ($mm),NVDA Operating Expenses ($mm),NVDA Operating Income ($mm),...,NVDA Operating Cash Flow ($mm),NVDA Capital Expenditures ($mm),NVDA Free Cash Flow ($mm),NVDA Stock-Based Compensation ($mm),NVDA Data Center Revenue ($mm),NVDA Data Center / Revenue,NVDA Operating Cash Flow Margin,NVDA Total Assets ($mm),NVDA Total Liabilities ($mm),NVDA Shareholders Equity ($mm)
0,2016 Q1,2016,1,1305,554,751,346,159,506,245,...,318,55,263,53,143,0.109579,0.243678,7108,2912,4196
1,2016 Q2,2016,2,1428,602,826,350,157,509,317,...,201,32,169,58,151,0.105742,0.140756,7461,2981,4480
2,2016 Q3,2016,3,2004,821,1183,373,171,544,639,...,432,38,394,65,240,0.119760,0.215569,9612,4288,5324
3,2016 Q4,2016,4,2173,870,1303,394,176,570,733,...,721,51,670,71,296,0.136217,0.331799,9841,4079,5762
4,2017 Q1,2017,1,1937,787,1150,411,185,596,554,...,282,54,228,76,409,0.211151,0.145586,9410,3278,6132
5,2017 Q2,2017,2,2230,928,1302,416,198,614,688,...,705,54,651,81,416,0.186547,0.316143,9402,3429,5973
6,2017 Q3,2017,3,2636,1067,1569,462,212,674,895,...,1157,69,1088,107,501,0.190061,0.438923,9830,3478,6352
7,2017 Q4,2017,4,2911,1110,1801,507,221,728,1073,...,1358,416,942,127,606,0.208176,0.466506,11241,3770,7471
8,2018 Q1,2018,1,3207,1139,2068,542,231,773,1295,...,1445,118,1327,129,701,0.218584,0.450577,11460,3743,7717
9,2018 Q2,2018,2,3123,1148,1975,581,237,818,1157,...,913,129,784,132,760,0.243356,0.292347,12882,4087,8795


In [42]:

display(df.describe())


,Year,Quarter Number,NVDA Revenue ($mm),NVDA Cost of Revenue ($mm),NVDA Gross Profit ($mm),NVDA R&D Expense ($mm),NVDA SG&A Expense ($mm),NVDA Operating Expenses ($mm),NVDA Operating Income ($mm),NVDA Net Income ($mm),...,NVDA Operating Cash Flow ($mm),NVDA Capital Expenditures ($mm),NVDA Free Cash Flow ($mm),NVDA Stock-Based Compensation ($mm),NVDA Data Center Revenue ($mm),NVDA Data Center / Revenue,NVDA Operating Cash Flow Margin,NVDA Total Assets ($mm),NVDA Total Liabilities ($mm),NVDA Shareholders Equity ($mm)
count,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,...,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000
mean,2020.761905,2.452381,16547.952381,4739.119048,11808.833333,1867.928571,559.261905,2459.476190,9349.357143,8829.000000,...,7227.785714,489.904762,6737.880952,637.500000,13373.261905,0.494073,0.367657,57460.500000,18577.642857,38882.857143
std,3.074756,1.130560,23367.481965,6007.915337,17475.319176,1714.408714,348.551152,2060.631329,15519.013026,15049.750354,...,11158.260653,597.124198,10682.902029,574.228401,22075.140969,0.287016,0.135007,69910.246315,18547.589718,51807.868484
min,2016.000000,1.000000,1305.000000,554.000000,751.000000,346.000000,157.000000,506.000000,245.000000,208.000000,...,201.000000,32.000000,-138.000000,53.000000,143.000000,0.105742,0.066093,7108.000000,2912.000000,4196.000000
25%,2018.000000,1.250000,2936.750000,1079.500000,1829.750000,615.500000,259.500000,875.500000,699.250000,662.000000,...,910.000000,128.250000,761.500000,144.000000,684.500000,0.244761,0.276060,13383.250000,4208.500000,9375.250000
50%,2021.000000,2.000000,5796.000000,2125.000000,3166.500000,1199.000000,541.500000,1722.000000,1346.500000,1375.000000,...,1603.500000,263.500000,1360.000000,447.000000,2200.000000,0.405155,0.400735,34723.000000,14428.000000,19960.500000
75%,2023.000000,3.000000,21107.250000,5164.000000,15943.250000,2422.250000,706.250000,3417.000000,12814.750000,11524.500000,...,10457.250000,500.000000,10197.250000,990.250000,17425.000000,0.824405,0.462524,62833.000000,22576.000000,40549.750000
max,2026.000000,4.000000,96221.000000,24079.000000,72142.000000,7054.000000,1354.000000,8408.000000,63734.000000,59688.000000,...,50344.000000,2677.000000,48587.000000,2026.000000,89000.000000,0.924954,0.622169,320272.000000,91288.000000,228984.000000


In [40]:

df.to_csv("NVDA_Final_Dataset.csv", index=False)

print("Dataset saved successfully.")


Dataset saved successfully.


## Data Exploration (EDA) -- Mikayla

In [ ]:
# initial inspection

In [ ]:
## summary statistics

In [ ]:
# visualizations of key trends & distributions

In [ ]:
# identification of potential issues (missing data, outliers, etc.)

## Data Cleaning & Preprocessing  -- Beck

In [ ]:
# Handling missing values, duplicates, and outliers


In [ ]:
# Encoding categorical variables


In [ ]:
# Scaling or normalization if required


In [ ]:
# Feature engineering (new calculated columns, transformations, aggregations)

## Modeling Approach - Mikayla

###Rationale for model selection (why you chose each algorithm)

### Description of 2–3 models to be tested

### Train/test split or cross-validation approach

## Model Implementation 1 - (model type) -- Nathan

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

 ## Model Implementation 2 - (model type) -- Beck

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

## Model Implementation 3 - (model type) -- Billy

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

## Model Evaluation -- Mikayla

In [ ]:
# Evaluate Model 1

In [ ]:
# Evaluate Model 2

In [ ]:
# Evaluate Model 3

In [ ]:
# Comparison of models

### Interpretation of model results & evaluation:


## Conclusions & Recommendations - Billy

### Best model choice and why

### Clear business recommendations

### Suggestions for next steps or further analysis

## Final Descriptive Charts - Nathan

Make sure all charts are labeled, correctly outputted, and have ocnsistent formatting

In [ ]:
# chart 1

In [ ]:
# chart 2

In [ ]:
# chart 3

In [ ]:
# chart 4

In [ ]:
# chart 5

In [ ]:
# chart 6 (optional)